# 04 Batch Wikipedia Extract Fallback Enrichment

This notebook completes the description rebuild at dataset scale. It keeps the high-quality Plot/Synopsis/Premise rows from `03_extensive_description_rebuild.ipynb`, then fills remaining source-matched titles using batched English Wikipedia plaintext extracts.

This is still source-grounded because pages are matched through IMDb `tconst -> Wikidata P345 -> English Wikipedia`, not through loose title search.

The output is the main enriched training dataset:

`data/cleaned_watch_data_phase2_enriched.csv`

Final-product role: this gives the modeling notebook enough consistent story text to create stable topics, labels, facets, and recommendation features for the backend catalog.


In [ ]:
from pathlib import Path
from urllib.parse import unquote, urlparse
import re
import time

import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 80)

DATA_DIR = Path("data")
BASE_DATA_PATH = DATA_DIR / "cleaned_watch_data.csv"
SOURCE_MAPPING_PATH = DATA_DIR / "extensive_wikidata_wikipedia_sources.csv"
LONG_PLOTS_PATH = DATA_DIR / "extensive_long_plot_summaries.csv"
BATCH_EXTRACTS_PATH = DATA_DIR / "extensive_wikipedia_full_extract_fallbacks.csv"
FINAL_ENRICHED_PATH = DATA_DIR / "cleaned_watch_data_extensive_enriched.csv"
BERTOPIC_COMPAT_PATH = DATA_DIR / "cleaned_watch_data_phase2_enriched.csv"

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "MovieMoodRecommenderStudentProject/2.0 (batch source-grounded description enrichment)"
MAX_EXTRACT_WORDS = 900
MIN_GOOD_MODELING_WORDS = 140
BATCH_SIZE = 25
CHECKPOINT_EVERY_BATCHES = 10
REQUEST_PAUSE_SECONDS = 0.35


In [ ]:
def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))


def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"\[[^\]]*\]", " ", str(text))
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def truncate_words(text, max_words):
    words = str(text).split()
    return " ".join(words[:max_words]) if len(words) > max_words else str(text)


def batched(values, batch_size):
    values = list(values)
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]


def request_json(session, params, retries=3, timeout=25):
    last_error = None
    for attempt in range(retries):
        try:
            response = session.post(WIKIPEDIA_API_URL, data=params, timeout=timeout)
            if response.status_code in {429, 500, 502, 503, 504}:
                time.sleep((attempt + 1) * 1.5)
                continue
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep((attempt + 1) * 1.5)
    raise RuntimeError(f"Wikipedia batch request failed: {last_error}")


## 1. Fetch Batched Full Extracts

This uses the already-built source mapping file. It resumes from `extensive_wikipedia_full_extract_fallbacks.csv` if present.

In [ ]:
base = pd.read_csv(BASE_DATA_PATH).reset_index(drop=True)
source_map = pd.read_csv(SOURCE_MAPPING_PATH)
source_map = source_map[source_map["wikipedia_title"].fillna("").astype(str).str.strip().ne("")].copy()
source_map = source_map.drop_duplicates("tconst")
# The mapping cache may contain rows fetched from Wikidata without local display metadata.
# Reattach canonical project metadata by tconst.
metadata_cols = ["tconst", "primary_title", "original_title", "display_title", "release_year", "genres", "description"]
source_map = source_map.drop(columns=[c for c in metadata_cols if c in source_map.columns and c != "tconst"], errors="ignore")
source_map = source_map.merge(base[metadata_cols], on="tconst", how="left")

if LONG_PLOTS_PATH.exists():
    long_plots = pd.read_csv(LONG_PLOTS_PATH)
    long_plot_tconsts = set(long_plots["tconst"].astype(str))
else:
    long_plots = pd.DataFrame()
    long_plot_tconsts = set()

if BATCH_EXTRACTS_PATH.exists() and BATCH_EXTRACTS_PATH.stat().st_size > 0:
    existing_extracts = pd.read_csv(BATCH_EXTRACTS_PATH)
else:
    existing_extracts = pd.DataFrame(columns=[
        "tconst", "primary_title", "release_year", "wikidata_id", "wikidata_label", "wikipedia_title",
        "wikipedia_url", "fallback_extract_text", "fallback_extract_word_count", "fallback_match_method", "license_note"
    ])

completed = set(existing_extracts["tconst"].astype(str)) | long_plot_tconsts
candidates = source_map[~source_map["tconst"].astype(str).isin(completed)].copy()

print("Base rows:", len(base))
print("Source-mapped rows:", len(source_map))
print("Existing plot/synopsis rows:", len(long_plot_tconsts))
print("Existing fallback extracts:", len(existing_extracts))
print("Remaining batch-extract candidates:", len(candidates))

display(candidates[["tconst", "primary_title", "release_year", "wikipedia_title"]].head())


In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

extract_rows = existing_extracts.to_dict(orient="records")

for batch_number, (_, batch_df) in enumerate(candidates.groupby(np.arange(len(candidates)) // BATCH_SIZE), start=1):
    titles = batch_df["wikipedia_title"].dropna().astype(str).tolist()
    if not titles:
        continue

    try:
        payload = request_json(session, {
            "action": "query",
            "prop": "extracts",
            "explaintext": "1",
            "exsectionformat": "plain",
            "exchars": "7000",
            "redirects": "1",
            "format": "json",
            "titles": "|".join(titles),
        })
        query = payload.get("query", {})
        pages = query.get("pages", {})
        title_to_extract = {}
        aliases = {}

        for page in pages.values():
            title = page.get("title", "")
            extract = clean_text(page.get("extract", ""))
            if extract:
                title_to_extract[title] = extract

        for item in query.get("normalized", []):
            aliases[item.get("from", "")] = item.get("to", "")
        for item in query.get("redirects", []):
            aliases[item.get("from", "")] = item.get("to", "")
        for src, dst in aliases.items():
            if dst in title_to_extract:
                title_to_extract[src] = title_to_extract[dst]

        for _, row in batch_df.iterrows():
            title = row["wikipedia_title"]
            extract = title_to_extract.get(title, "")
            if not extract:
                continue
            extract = truncate_words(extract, MAX_EXTRACT_WORDS)
            extract_rows.append({
                "tconst": row["tconst"],
                "primary_title": row.get("primary_title", ""),
                "release_year": row.get("release_year", ""),
                "wikidata_id": row.get("wikidata_id", ""),
                "wikidata_label": row.get("wikidata_label", ""),
                "wikipedia_title": title,
                "wikipedia_url": row.get("enwiki_url", ""),
                "fallback_extract_text": extract,
                "fallback_extract_word_count": word_count(extract),
                "fallback_match_method": "wikidata_p345_to_enwiki_batch_full_extract",
                "license_note": "Source-grounded plaintext extracted from English Wikipedia page linked to matching Wikidata IMDb ID (P345). Review attribution requirements before redistribution.",
            })
    except Exception as exc:
        print(f"Batch {batch_number} failed: {exc}")

    if batch_number % CHECKPOINT_EVERY_BATCHES == 0:
        pd.DataFrame(extract_rows).drop_duplicates("tconst", keep="last").to_csv(BATCH_EXTRACTS_PATH, index=False)
        print(f"Checkpoint batch {batch_number}; fallback rows: {len(pd.DataFrame(extract_rows).drop_duplicates('tconst'))}", flush=True)

    time.sleep(REQUEST_PAUSE_SECONDS)

fallback_extracts = pd.DataFrame(extract_rows).drop_duplicates("tconst", keep="last")
fallback_extracts.to_csv(BATCH_EXTRACTS_PATH, index=False)
print("Saved fallback extracts:", BATCH_EXTRACTS_PATH, fallback_extracts.shape)
display(fallback_extracts.tail())


## 2. Build Final Training Dataset

Preference order:

1. exact Plot/Synopsis/Premise extraction from notebook 03
2. batch full-page Wikipedia extract fallback
3. original cleaned description, flagged for manual research if still weak

In [ ]:
def build_final_enriched_dataset():
    base = pd.read_csv(BASE_DATA_PATH).reset_index(drop=True)
    base["row_id"] = np.arange(len(base))
    base["description_word_count"] = base["description"].fillna("").map(word_count)

    if LONG_PLOTS_PATH.exists() and LONG_PLOTS_PATH.stat().st_size > 0:
        long_plots = pd.read_csv(LONG_PLOTS_PATH)
    else:
        long_plots = pd.DataFrame(columns=["tconst"])

    if BATCH_EXTRACTS_PATH.exists() and BATCH_EXTRACTS_PATH.stat().st_size > 0:
        fallback = pd.read_csv(BATCH_EXTRACTS_PATH)
    else:
        fallback = pd.DataFrame(columns=["tconst"])

    plot_cols = [
        "tconst", "wikidata_id", "wikidata_label", "wikipedia_title", "wikipedia_url", "source_section",
        "enrichment_text", "enrichment_word_count", "match_method", "license_note"
    ]
    plot_cols = [c for c in plot_cols if c in long_plots.columns]
    plot_data = long_plots[plot_cols].drop_duplicates("tconst") if not long_plots.empty else long_plots

    fallback_cols = [
        "tconst", "wikidata_id", "wikidata_label", "wikipedia_title", "wikipedia_url",
        "fallback_extract_text", "fallback_extract_word_count", "fallback_match_method", "license_note"
    ]
    fallback_cols = [c for c in fallback_cols if c in fallback.columns]
    fallback_data = fallback[fallback_cols].drop_duplicates("tconst") if not fallback.empty else fallback

    out = base.merge(plot_data, on="tconst", how="left")
    out = out.merge(fallback_data, on="tconst", how="left", suffixes=("", "_fallback"))

    if "fallback_extract_text" not in out.columns:
        out["fallback_extract_text"] = ""
    if "enrichment_text" not in out.columns:
        out["enrichment_text"] = ""

    has_plot = out["enrichment_text"].fillna("").astype(str).str.strip().ne("")
    has_fallback = out["fallback_extract_text"].fillna("").astype(str).str.strip().ne("")

    out["final_enrichment_text"] = np.where(has_plot, out["enrichment_text"], out["fallback_extract_text"])
    out["final_enrichment_source_type"] = np.select(
        [has_plot, has_fallback],
        ["plot_or_synopsis_section", "wikipedia_full_extract_fallback"],
        default="original_description_only",
    )
    out["has_phase2_enrichment"] = has_plot | has_fallback
    out["enrichment_text"] = out["final_enrichment_text"]
    out["enrichment_word_count"] = out["enrichment_text"].fillna("").map(word_count)
    out.loc[~out["has_phase2_enrichment"], "enrichment_word_count"] = np.nan

    def modeling_description(row):
        parts = []
        original = clean_text(row.get("description", ""))
        enriched = clean_text(row.get("enrichment_text", ""))
        if original:
            parts.append(f"Current description: {original}")
        if enriched:
            parts.append(f"Source-grounded expanded description: {truncate_words(enriched, MAX_EXTRACT_WORDS)}")
        return " ".join(parts).strip()

    out["modeling_description"] = out.apply(modeling_description, axis=1)
    out["modeling_description_word_count"] = out["modeling_description"].map(word_count)
    out["needs_manual_research"] = out["modeling_description_word_count"] < MIN_GOOD_MODELING_WORDS
    out["description_rebuild_status"] = np.select(
        [has_plot, has_fallback, out["needs_manual_research"]],
        ["source_grounded_plot_or_synopsis_added", "source_grounded_full_extract_added", "needs_manual_research_no_verified_long_source"],
        default="original_description_sufficient_but_unexpanded",
    )

    out.to_csv(FINAL_ENRICHED_PATH, index=False)
    out.to_csv(BERTOPIC_COMPAT_PATH, index=False)
    return out

rebuilt_df = build_final_enriched_dataset()
print("Saved:", FINAL_ENRICHED_PATH)
print("Saved:", BERTOPIC_COMPAT_PATH)
print("Rows:", len(rebuilt_df))
print("Rows with enrichment:", int(rebuilt_df["has_phase2_enrichment"].sum()))
print("Rows still needing manual research:", int(rebuilt_df["needs_manual_research"].sum()))
print(rebuilt_df["final_enrichment_source_type"].value_counts().to_string())

display(rebuilt_df[[
    "tconst", "primary_title", "release_year", "genres", "description_word_count", "has_phase2_enrichment",
    "final_enrichment_source_type", "enrichment_word_count", "modeling_description_word_count", "needs_manual_research"
]].head(20))


## 3. Inspect Known Problem Titles

In [ ]:
problem_titles = ["Inside Out", "Invincible", "The Bad Guys", "Demon Slayer: Kimetsu no Yaiba", "The Promised Neverland", "Home", "The Girl Who Leapt Through Time"]
cols = [
    "tconst", "content_type", "primary_title", "release_year", "genres", "description_word_count",
    "has_phase2_enrichment", "final_enrichment_source_type", "enrichment_word_count",
    "modeling_description_word_count", "needs_manual_research", "wikipedia_title", "description", "enrichment_text"
]
cols = [c for c in cols if c in rebuilt_df.columns]
display(rebuilt_df[rebuilt_df["primary_title"].isin(problem_titles)][cols].sort_values(["primary_title", "release_year"]))
